# 01 · Exploratory data analysis

Run this **after** `uv run dvc repro` (or `uv run dvc pull`). It reads the pipeline outputs. It doesn't recompute them.

| Section | Question it answers |
|---|---|
| 1 | What does each Naukri snapshot look like after cleaning? |
| 2 | How many skills per posting, and how big must the skill vocabulary be? |
| 3 | Which synonyms are still not merged by `configs/skill_aliases.yaml`? |
| 4 | Which skills rose or fell between 2019 and 2025 (real drift)? |
| 5 | Experience: what do postings ask for? |
| 6 | How good are the title → role rules? |
| 7 | **Which roles should be classes?** Comparing all snapshots |
| 8 | Class balance in the training split |
| 9 | Do the role skill profiles make sense? (rule-bug detector) |

The automated profiling reports (one per snapshot) are in `reports/profiling/*.html`.

In [ ]:
import os, re, json, collections
from pathlib import Path
if Path.cwd().name == "notebooks":
    os.chdir("..")   # run from the repo root so paths match dvc.yaml

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.max_colwidth", 60)
pd.set_option("display.width", 200)

postings = pd.read_parquet("data/interim/postings.parquet")
labelled = pd.read_parquet("data/interim/labelled.parquet")
train = pd.read_parquet("data/processed/train.parquet")
support = pd.read_csv("reports/taxonomy/role_support.csv")
profiles = json.loads(Path("models/skill_profiles.json").read_text())
for df in (postings, labelled, train):
    df["skills"] = df["skills"].map(list)
SNAPS = sorted(postings["snapshot"].unique())
print({s: int((postings.snapshot == s).sum()) for s in SNAPS})

## 1 · The snapshots after cleaning
`reports/ingest_summary.json`. 2017 has no skill tags (its `skills` column was really the functional area), so it's used only for titles, labels and education.

In [ ]:
pd.DataFrame(json.loads(Path("reports/ingest_summary.json").read_text())).T

## 2 · Skills per posting and vocabulary size
Users will type ~5 skills; postings list ~8. The model only knows the top‑K skills (`features.top_k_skills`). How many postings keep ≥ 3 known skills as K grows?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for s in [2019, 2022, 2025]:
    postings.loc[postings.snapshot == s, "n_skills"].clip(upper=25).plot.hist(bins=25, alpha=.5, ax=axes[0], label=str(s))
axes[0].set(title="Skills per posting (clipped at 25)", xlabel="n_skills"); axes[0].legend()

p25 = postings[(postings.snapshot == 2025) & (postings.n_skills > 0)]
counts = p25["skills"].explode().value_counts()
ks = [100, 250, 500, 750, 1000, 1500, 2000, 3000, 5000]
cov = [p25["skills"].map(lambda l, v=set(counts.index[:k]): sum(x in v for x in l) >= 3).mean() for k in ks]
axes[1].plot(ks, cov, marker="o"); axes[1].set(title="2025: postings with ≥3 skills inside the top-K vocabulary", xlabel="K", ylabel="share")
plt.tight_layout(); plt.show()
print(f"{len(counts):,} distinct skills in 2025; coverage by K:", dict(zip(ks, np.round(cov, 3))))

## 3 · Synonyms still not merged
Groups tags that become identical once spaces, punctuation and a trailing `js` are removed. Anything here that really is one skill belongs in `configs/skill_aliases.yaml`.

In [ ]:
c = collections.Counter(postings.loc[postings.snapshot != 2017, "skills"].explode().dropna())
def squash(t):
    k = re.sub(r"[^a-z0-9+#]", "", t)
    return re.sub(r"js$", "", k) if len(k) > 4 else k
groups = collections.defaultdict(list)
for t, _ in c.most_common(4000):
    groups[squash(t)].append(t)
left = sorted((g for g in groups.values() if len(g) > 1), key=lambda g: -sum(c[t] for t in g))
pd.DataFrame({"variants (count)": [" | ".join(f"{t} ({c[t]})" for t in g) for g in left[:30]]})

## 4 · Skill drift 2019 → 2025
Share of postings asking for each skill. This is the *real* drift the monitoring phase will detect.

In [ ]:
def shares(s):
    sub = postings[(postings.snapshot == s) & (postings.n_skills > 0)]
    return sub["skills"].explode().value_counts() / len(sub)
sh = pd.DataFrame({y: shares(y) for y in [2019, 2022, 2025]}).fillna(0)
big = sh[(sh[2019] >= .005) | (sh[2025] >= .005)].copy()
big["change_pp"] = (big[2025] - big[2019]) * 100
fig, ax = plt.subplots(figsize=(10, 7))
top = pd.concat([big.nsmallest(12, "change_pp"), big.nlargest(12, "change_pp")])
top["change_pp"].plot.barh(ax=ax, color=np.where(top["change_pp"] > 0, "tab:green", "tab:red"))
ax.set(title="Biggest changes in share of postings, 2019 → 2025 (percentage points)", xlabel="pp"); plt.tight_layout(); plt.show()
(big.sort_values("change_pp")[[2019, 2022, 2025]] * 100).round(2).iloc[np.r_[0:8, -8:0]]

## 5 · Experience asked for
The model uses the posting's **minimum** years (`exp_min`), since a user enters one number.

In [ ]:
postings.groupby("snapshot")[["exp_min", "exp_max"]].describe().round(1)

## 6 · How good are the title → role rules?
2017 and 2019 carry Naukri's own functional area. `reports/label_quality.json` has coverage and family agreement. The CSVs show where to improve `configs/role_rules.yaml`.

In [ ]:
pd.DataFrame(json.loads(Path("reports/label_quality.json").read_text())).T

In [ ]:
pd.read_csv("reports/labels/disagreements.csv").head(15)

In [ ]:
pd.read_csv("reports/labels/unlabelled_titles.csv").head(25)

## 7 · Which roles should be classes? Comparing all snapshots

Each candidate role's usable postings per snapshot (`reports/taxonomy/role_support.csv`). The pipeline applies the thresholds in `params.yaml → taxonomy`:
- **stable**: enough postings in 2025 *and* 2019, so we can measure drift for it;
- **emerging**: enough in 2025 only (new or fast-growing roles);
- **excluded**: too rare in 2025 to learn.

Things to look for:
- roles near the thresholds;
- roles whose Naukri 2019 label share (`naukri_2019_top_role_share`) is very low, meaning the rule may be catching unrelated titles;
- 2022, which is IT-heavy: non-tech roles are nearly absent there, so it can't be used as a general snapshot.

In [ ]:
cols = ["role", "family", "n_2017", "n_2019", "n_2022", "n_2025", "share_change_ref_to_train",
        "naukri_2019_top_role", "naukri_2019_top_role_share", "status"]
support[cols].style.background_gradient(subset=["share_change_ref_to_train"], cmap="RdYlGn", vmin=0, vmax=3)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 8))
for st, colr in [("stable", "tab:blue"), ("emerging", "tab:orange"), ("excluded", "tab:gray")]:
    d = support[support.status == st]
    ax.scatter(d.share_2019.clip(lower=2e-5), d.share_2025.clip(lower=2e-5), label=f"{st} ({len(d)})", c=colr, s=25)
lim = [2e-5, .2]
ax.plot(lim, lim, "k--", lw=.8, label="same share in both years")
ax.set(xscale="log", yscale="log", xlim=lim, ylim=lim, xlabel="share of 2019 postings", ylabel="share of 2025 postings",
       title="Role share, 2019 vs 2025 (above the line = growing)")
for r in support.assign(ch=support.share_change_ref_to_train).sort_values("ch").iloc[np.r_[0:5, -6:0]].itertuples():
    ax.annotate(r.role, (max(r.share_2019, 2e-5), r.share_2025), fontsize=8)
ax.legend(); plt.show()
print(json.loads(Path("reports/taxonomy_summary.json").read_text()))

## 8 · Class balance in the training split

In [ ]:
sizes = train["role"].value_counts()
fig, ax = plt.subplots(figsize=(10, 9))
pd.concat([sizes.head(15), sizes.tail(15)]).sort_values().plot.barh(ax=ax)
ax.set(title=f"Training postings per role: 15 largest and 15 smallest of {sizes.size}", xlabel="postings")
plt.tight_layout(); plt.show()
print(json.loads(Path("reports/split_summary.json").read_text()))

## 9 · Do the role skill profiles make sense?
A role's core skills should obviously belong to it. If they don't (e.g. a *Recruiter* whose top skills are `voice process`, `bpo`), a rule is catching the wrong titles. Roles with a very flat profile (top skill in < 15% of postings) are the first to check.

In [ ]:
rows = []
for role, r in profiles["roles"].items():
    top = list(r["top_skills"].items())[:6]
    rows.append({"role": role, "family": r["family"], "n": r["n_postings"], "top_share": top[0][1] if top else 0,
                 "top skills": ", ".join(f"{s} {v:.0%}" for s, v in top)})
prof = pd.DataFrame(rows).sort_values("top_share")
prof.head(20)

## Decisions & notes (fill in as a team)

- Taxonomy thresholds agreed: `min_train_postings = …`, `min_reference_postings = …`. Why:
- Rules changed after reviewing sections 6, 7 and 9:
- Aliases added after section 3:
- Open questions: